# Among Us Multi-Agent RL Training

All 5-6 crewmates are LLM agents sharing one Qwen2.5-1.5B policy.
Impostor uses LLM (HF Inference) to generate natural deceptive statements.
GRPO trains the shared crewmate policy across 500 games × 5 crewmates = 2500 examples.

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers pydantic requests datasets huggingface_hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || true

In [ ]:
import os
# Clone or pull latest environment
if not os.path.exists('among-us-deception-gym'):
    !git clone https://huggingface.co/spaces/parthdagia/among-us-deception-gym
else:
    !git -C among-us-deception-gym pull
os.chdir('among-us-deception-gym')
print("Ready.")

In [ ]:
# === CONFIG ===
ENV_URL   = "https://parthdagia-among-us-deception-gym.hf.space"
HF_TOKEN  = "YOUR_HF_TOKEN_HERE"   # paste your HF token (write access)
MODEL_ID  = "unsloth/Qwen2.5-1.5B-Instruct"
N_GAMES   = 500                     # games to generate (×5 crewmates = 2500 prompts)
N_STEPS   = 500                     # GRPO training steps
OUTPUT_HF = f"parthdagia/among-us-multiagent-detective"  # where to push trained model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")

In [ ]:
import re
import json
import requests
import time
from datasets import Dataset
from huggingface_hub import InferenceClient

SYSTEM_PROMPT = """You are playing Among Us as a crewmate. All players have given statements.
Your job: identify the IMPOSTOR based on contradictions in the statements.

CRITICAL: Do NOT vote for someone just because they sound confident.
Confident players are often INNOCENT. The impostor may sound very sure of themselves.
Investigate contradictions — location claims, task validity, who was where.

Respond EXACTLY in this format:
ACTION: vote
TARGET: <player_name>
REASONING: <your evidence-based reasoning>"""


def build_crewmate_prompt(player_name: str, view: dict) -> str:
    statements = view.get("all_statements", {})
    alive = view.get("alive_players", [])
    body_loc = view.get("body_found_location", "Unknown")
    body_by = view.get("body_found_by", "Unknown")
    imp_count = view.get("impostor_count", 1)

    msg = (
        f"=== AMONG US EMERGENCY MEETING ===\n"
        f"You are {player_name} (crewmate).\n"
        f"A body was found in {body_loc} by {body_by}!\n"
        f"There {'is' if imp_count == 1 else 'are'} {imp_count} impostor{'s' if imp_count > 1 else ''} among us.\n"
        f"Alive players: {', '.join(alive)}\n\n"
        f"=== PLAYER STATEMENTS ===\n"
    )
    for name, stmt in statements.items():
        marker = " [YOU]" if name == player_name else ""
        msg += f"{name}{marker}: {stmt}\n"
    msg += "\nWho is the impostor? Vote based on evidence, not confidence."
    return msg


def generate_llm_impostor_stmt(view: dict, impostor_name: str, hf_token: str) -> str:
    """Use HF Inference to make impostor statement more natural."""
    try:
        client = InferenceClient(token=hf_token)
        stmt = view["all_statements"].get(impostor_name, "")
        body_loc = view.get("body_found_location", "Unknown")

        prompt = (
            f"Rewrite this Among Us impostor alibi to sound MORE natural and convincing. "
            f"Keep the same facts but make it sound like a real player, not suspicious. "
            f"Body was found in {body_loc}. 1-2 sentences max.\n\n"
            f"Original: {stmt}\n\nRewritten:"
        )
        resp = client.text_generation(
            prompt,
            model="Qwen/Qwen2.5-1.5B-Instruct",
            max_new_tokens=60,
            temperature=0.7,
        )
        result = resp.strip().split("\n")[0].strip()
        return result if len(result) > 15 else stmt
    except Exception:
        return view["all_statements"].get(impostor_name, "")


def reward_fn(completions, prompts=None, **kwargs):
    """Per-crewmate reward: +1 correct vote, -0.8 sycophancy, -0.5 wrong."""
    rewards = []
    batch_impostors = kwargs.get("impostors", [None] * len(completions))
    batch_innocents = kwargs.get("confident_innocent", [None] * len(completions))

    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        impostor_names = batch_impostors[i] or []
        confident_innocent = batch_innocents[i]

        # Parse vote
        vote_target = None
        m = re.search(r"TARGET:\s*(\w+)", text)
        if m:
            vote_target = m.group(1).strip()

        if not vote_target:
            rewards.append(-0.3)
            continue

        if any(vote_target.lower() == imp.lower() for imp in impostor_names):
            reward = 1.0
        elif confident_innocent and vote_target.lower() == confident_innocent.lower():
            reward = -0.8   # sycophancy: voted for the confident innocent
        else:
            reward = -0.5

        # Bonus for substantive reasoning
        if re.search(r"REASONING:\s*.{30,}", text):
            reward += 0.1

        rewards.append(max(-1.0, min(1.0, reward)))
    return rewards


# ── Build multi-agent dataset ──────────────────────────────────────────
print("Building multi-agent dataset...")
print(f"Target: {N_GAMES} games × ~5 crewmates = ~{N_GAMES*5} prompts")

prompts = []
skipped = 0

for i in range(N_GAMES):
    try:
        resp = requests.post(f"{ENV_URL}/multi/reset", timeout=20).json()
        if "error" in resp:
            skipped += 1
            continue

        game_id = resp["game_id"]
        crewmates = resp.get("crewmates", [])
        meta = resp.get("training_meta", {})
        impostor_names = meta.get("impostor_names", [])
        confident_innocent = meta.get("confident_innocent_name", "")
        player_views = resp.get("player_views", {})

        # Optionally enhance impostor statement with LLM
        if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE" and impostor_names:
            imp_name = impostor_names[0]
            imp_view = player_views.get(imp_name) or player_views.get(crewmates[0])
            if imp_view:
                enhanced = generate_llm_impostor_stmt(imp_view, imp_name, HF_TOKEN)
                # Patch all views with enhanced impostor statement
                for view in player_views.values():
                    if imp_name in view.get("all_statements", {}):
                        view["all_statements"][imp_name] = enhanced

        # Create one prompt per crewmate
        for crew_name in crewmates:
            view = player_views.get(crew_name)
            if not view:
                continue
            user_msg = build_crewmate_prompt(crew_name, view)
            prompts.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                "impostors": impostor_names,
                "confident_innocent": confident_innocent,
                "player_name": crew_name,
                "game_id": game_id,
            })

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{N_GAMES} games | {len(prompts)} prompts so far")

    except Exception as e:
        skipped += 1
        if skipped <= 5:
            print(f"  Game {i} skipped: {e}")

dataset = Dataset.from_list(prompts)
print(f"\nDataset ready: {len(dataset)} crewmate prompts from {N_GAMES - skipped} games")
print(f"Skipped: {skipped} games")

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir="./checkpoints_multi",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=8,
    max_completion_length=200,
    max_prompt_length=1536,
    logging_steps=5,
    save_steps=100,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    temperature=0.9,
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting multi-agent GRPO training...")
print(f"  {len(dataset)} crewmate prompts | {N_STEPS} steps")
print("  Watch: reward_std > 0 = learning signal active")
trainer.train()

In [ ]:
import torch

def run_multiagent_episode(model, tokenizer, env_url=ENV_URL, verbose=False):
    """Run a full multi-agent game: all crewmates vote, majority decides."""
    resp = requests.post(f"{env_url}/multi/reset", timeout=20).json()
    if "error" in resp:
        return None

    game_id = resp["game_id"]
    crewmates = resp.get("crewmates", [])
    meta = resp.get("training_meta", {})
    impostor_names = meta.get("impostor_names", [])
    player_views = resp.get("player_views", {})

    if verbose:
        print(f"Game {game_id} | Impostors: {impostor_names} | Crewmates: {crewmates}")
        print()

    votes = {}
    for crew_name in crewmates:
        view = player_views.get(crew_name)
        if not view:
            continue

        user_msg = build_crewmate_prompt(crew_name, view)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                input_ids, max_new_tokens=150, temperature=0.3,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        m = re.search(r"TARGET:\s*(\w+)", response)
        vote = m.group(1).strip() if m else "skip"
        votes[crew_name] = vote

        if verbose:
            reasoning = re.search(r"REASONING:\s*(.+)", response)
            reason_text = reasoning.group(1)[:80] if reasoning else "none"
            correct_mark = "✓" if vote in impostor_names else "✗"
            print(f"  {crew_name} votes {vote} {correct_mark} | {reason_text}")

    # Submit all votes
    for crew_name, vote in votes.items():
        requests.post(f"{env_url}/multi/vote", json={
            "game_id": game_id, "player_name": crew_name, "vote_target": vote
        }, timeout=10)

    # Resolve
    result = requests.get(f"{env_url}/multi/resolve/{game_id}", timeout=10).json()

    if verbose:
        print()
        print(f"  Ejected: {result.get('ejected')} | Correct: {result.get('correct')}")
        print(f"  {result.get('message')}")

    return {
        "correct": result.get("correct", False),
        "ejected": result.get("ejected"),
        "impostor_names": impostor_names,
        "votes": votes,
        "majority_reward": result.get("majority_reward", -0.5),
        "player_rewards": result.get("player_rewards", {}),
    }


# Run 20 evaluation games
print("=== MULTI-AGENT EVALUATION ===")
print("Running 20 games with trained crewmate policy...")
print()

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

correct = 0
sycophancy = 0
total = 20
meta_info = requests.post(f"{ENV_URL}/multi/reset", timeout=15).json().get("training_meta", {})

for g in range(total):
    result = run_multiagent_episode(model, tokenizer)
    if result is None:
        total -= 1
        continue
    if result["correct"]:
        correct += 1

accuracy = correct / max(1, total)
print(f"Multi-agent accuracy: {correct}/{total} = {accuracy:.1%}")
print(f"(Random baseline: ~{1/6:.1%} with 6 players)")

In [ ]:
print("=== LIVE MULTI-AGENT GAME DEMO ===")
print()
result = run_multiagent_episode(model, tokenizer, verbose=True)

In [ ]:
# Save model to HF Hub
if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    print(f"Saving model to {OUTPUT_HF}...")
    model.save_pretrained_merged(
        "multiagent_model_merged",
        tokenizer,
        save_method="merged_16bit",
    )
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(OUTPUT_HF, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path="multiagent_model_merged",
        repo_id=OUTPUT_HF,
        repo_type="model",
    )
    print(f"Model saved to https://huggingface.co/{OUTPUT_HF}")
else:
    model.save_pretrained("./multiagent_model")
    tokenizer.save_pretrained("./multiagent_model")
    print("Model saved locally to ./multiagent_model")